# Pilot 2 — exploratory rank-1 GSM8K SFT

**Question:** What happens to GSM8K accuracy and response length after a small SFT intervention?

We train Qwen2.5-1.5B-Instruct on 512 gold solutions using rank-1 LoRA, then evaluate a fresh matched untuned baseline and checkpoints 0/8/16/32/64 on the same 150 held-out questions. This pilot evaluates **GSM8K only**.

The historical SPAR result was accuracy 0.640 → 0.420 and mean response length 288 → 77 tokens. Its recipe is unavailable. This is a **new exploratory protocol**, with no directional prediction or binary collapse gate. Historical values are descriptive references.

The pipeline is **freeze inputs → inspect targets/plan → GPU preflight → matched baseline → train/resume → checkpoint evaluation → paired report**. CPU tests check pipeline logic; the actual Qwen/T4 checks below establish whether this runtime fits and is numerically stable. Local CPU success does not establish GPU success.

## 1. Choose the runtime and restore the repository

In Colab choose **Runtime → Change runtime type → T4 GPU**. This protocol uses unquantized FP32, training microbatch 1 × accumulation 8, evaluation batch 1, and gradient checkpointing. FP32 is slower and uses more memory than FP16; the preflight will stop if it does not fit. It will not silently change precision.

Mount Drive and use the same artifact root as Pilot 1 so the original item files remain available. The new repository checkout uses a separate local directory; your Pilot 1 checkout is preserved. The code revision is pinned for resume.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/mahadikprasad15/spar-pilot.git'
HARNESS_COMMIT = '3f367bfc70857ddc24145f2aea3663ad8104a59f'
REPO_DIR = Path('/content/spar-pilot2')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/SPAR/pilot1/artifacts')
PLAN_NAME = 'pilot2-sft-fp32-v1'
SOURCE_CONFIG = ARTIFACT_ROOT / 'plans/baseline-batch8-v1/gsm8k-0shot.config.json'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
assert not dirty, 'Preserve local edits before switching the pinned checkout.'
subprocess.run(['git', 'fetch', 'origin', HARNESS_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', HARNESS_COMMIT], cwd=REPO_DIR, check=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
SESSION_DIR = ARTIFACT_ROOT / 'notebook' / PLAN_NAME
SESSION_DIR.mkdir(parents=True, exist_ok=True)
print('Checkout:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip())
print('Persistent artifacts:', ARTIFACT_ROOT)
print('Source config exists:', SOURCE_CONFIG.exists())

## 2. Install the pinned environment — then restart the session

This installs the exact direct library versions for the new protocol. A Colab runtime may come with different Torch/Transformers versions; installing without restarting can leave old versions in memory.

**After this cell completes, choose Runtime → Restart session.** Then rerun section 1, skip this install cell, and continue at section 3. Do not disconnect/delete the runtime or run the whole notebook automatically across this restart.

Installation downloads libraries. Preparation downloads the pinned dataset and tokenizer; Qwen weights are downloaded at preflight. Existing model caches can be reused.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-pilot2.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(REPO_DIR)], check=True)
print('Installation complete. Restart the session, rerun section 1, then continue at section 3.')

## 3. Verify libraries, GPU and CPU tests

Expected: the pinned versions, one Tesla T4, and passing CPU tests. These tests download no model. A small real model is constructed locally for the TRL/PEFT integration test. Stop and inspect any failure before proceeding.

The old BF16 Pilot 1 scores are retained as references. We measure a fresh FP32 baseline under this exact environment instead of treating the earlier score as matched.

In [ ]:
import importlib.metadata
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(REPO_DIR))
from pilot_eval.sft_backend import PINS
import torch
for package, expected in PINS.items():
    actual = importlib.metadata.version(package)
    print(package, actual)
    assert actual.split('+')[0] == expected, f'{package} version mismatch; install/restart first.'
assert torch.cuda.is_available() and torch.cuda.device_count() == 1
print('GPU:', torch.cuda.get_device_name(0), 'capability:', torch.cuda.get_device_capability(0))
assert 'T4' in torch.cuda.get_device_name(0), 'This registered protocol selects T4.'
test_environment = dict(os.environ, HF_HUB_OFFLINE='1', HF_DATASETS_OFFLINE='1')
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR, env=test_environment, check=True)

## 4. Helpers and frozen preparation

`SOURCE_CONFIG` must point to an **untuned GSM8K config**, not the combined report. Its `items_path` must still exist within `ARTIFACT_ROOT`. If your source plan has another name, edit only that path in section 1. Preparation verifies the original tokenizer/template and item contents rather than choosing another evaluation set.

The training cohort is a seed-42 sample of 512 official train examples. The evaluation cohort stays the original 150 official test examples. Gold calculator annotations and `####` markers are kept. Targets include the assistant end-turn; question and padding tokens receive no loss. Complete sequences are checked without truncation.

Commands run as separate processes, write console logs to Drive, and show durable progress. Do not start another GPU command if an earlier child is still alive after a notebook interruption. Check the PID file and saved status first; the scripts also lock their runs.

In [ ]:
import time

def restore_state():
    from pilot_eval.sft import load_sft
    path = ARTIFACT_ROOT / 'plans' / PLAN_NAME / 'sft.config.json'
    config, rows, items = load_sft(path, ARTIFACT_ROOT)
    return path, config, rows, items

def run_command(arguments, label, status_path=None):
    command = [sys.executable, '-u', '-m', 'pilot_eval', *map(str, arguments), '--output-root', str(ARTIFACT_ROOT)]
    console_path = SESSION_DIR / (label + '.console.log')
    pid_path = SESSION_DIR / (label + '.process.json')
    if pid_path.exists():
        previous = json.loads(pid_path.read_text())
        if previous.get('running'):
            try:
                os.kill(previous['pid'], 0)
            except ProcessLookupError:
                pass
            else:
                raise RuntimeError(f"Earlier process may still be alive: PID {previous['pid']}. Inspect before restarting.")
    print('Starting:', label, flush=True)
    with console_path.open('a') as console:
        process = subprocess.Popen(command, cwd=REPO_DIR, stdout=console, stderr=subprocess.STDOUT)
        pid_path.write_text(json.dumps(dict(pid=process.pid, command=command, running=True)))
        while True:
            try:
                returncode = process.wait(timeout=10)
                break
            except subprocess.TimeoutExpired:
                status = None
                if status_path:
                    try:
                        status = json.loads(Path(status_path).read_text())
                    except (OSError, json.JSONDecodeError):
                        pass
                print(f"{label}: " + (f"{status['state']} {status['completed']}/{status['total']}" if status else f'process {process.pid} active'), flush=True)
    pid_path.write_text(json.dumps(dict(pid=process.pid, command=command, running=False, returncode=returncode)))
    if returncode:
        print('\n'.join(console_path.read_text().splitlines()[-25:]))
        raise RuntimeError(f'Command stopped. Inspect {console_path} and the run logs; settings were not changed.')
    print('Completed:', label, '| console:', console_path, flush=True)

assert SOURCE_CONFIG.is_file(), f'Select your saved untuned GSM8K config: {SOURCE_CONFIG}'
run_command(['sft-prepare', '--source-config', SOURCE_CONFIG, '--name', PLAN_NAME], 'prepare')
SFT_CONFIG, CONFIG, TRAIN_ROWS, EVAL_ITEMS = restore_state()
print('Training examples:', len(TRAIN_ROWS), 'evaluation examples:', len(EVAL_ITEMS))
print('Longest complete sequence:', CONFIG['max_length'])
print('Optimizer:', CONFIG['optimizer'])

## 5. Inspect what the model will learn and the exploratory plan

This is **teacher-forced SFT**: given the question and preceding gold tokens, train the adapter to predict the next gold token. It is not sampled solution generation or reward training.

Inspect the rendered target and supervision counts. Expected: the familiar Qwen system/user/assistant formatting, original gold solution, masked prompt and supervised completion ending. The stored exploratory plan freezes strict and flexible-v2 scorers, Wilson accuracy intervals, and 2,000 paired bootstrap draws with seed 42. There is no prediction or threshold to make the result pass.

Gold length is measured from gold text alone; training length includes the prompt and end-turn. Similar gold/generated lengths later are suggestive, not a causal explanation.

In [ ]:
SFT_CONFIG, CONFIG, TRAIN_ROWS, EVAL_ITEMS = restore_state()
row = TRAIN_ROWS[0]
print('TRAIN ID:', row['id'])
print('RENDERED TARGET:\n', row['rendered'])
print('Prompt tokens masked:', row['prompt_tokens'])
print('Completion tokens supervised:', sum(label != -100 for label in row['labels']))
print('Final supervised token ID:', row['labels'][-1])
print('Gold-only token count:', row['gold_tokens'])
print('\nEXPLORATORY PLAN:\n', json.dumps(json.loads((ARTIFACT_ROOT / CONFIG['analysis_plan_path']).read_text()), indent=2))

## 6. GPU memory/stability preflight

This loads the actual pinned Qwen model and attaches all 196 intended rank-1 LoRA modules. It checks zero initial write, frozen base weights, the longest complete example, eight accumulated backward passes, finite loss/gradients and optimizer-state allocation. The disposable optimizer update is restored exactly; preflight does not advance the scientific run.

Expected: saved passing evidence and memory usage. **An OOM or numerical/integrity error stops here.** Retrying cannot make an oversized sequence fit. Preserve logs; changing precision or settings requires an explicitly revised protocol, rather than editing an existing run.

In [ ]:
from pilot_eval.training import training_directory
SFT_CONFIG, CONFIG, _, _ = restore_state()
TRAIN_DIR = training_directory(ARTIFACT_ROOT, CONFIG)
run_command(['sft-train', '--config', SFT_CONFIG, '--preflight-only'], 'preflight', TRAIN_DIR / 'meta/status.json')
print(json.dumps(json.loads((TRAIN_DIR / 'results/preflight.json').read_text()), indent=2))

## 7. Fresh matched untuned baseline

Generate once on the fixed 150 items using FP32, greedy decoding, batch 1 and the saved 1024-token cap. Save every response and score the same text with strict and flexible-v2 extraction. No adapter is attached.

Greedy means choosing the highest-scoring next token at each step. Strict scoring demands the specified final-answer format; flexible-v2 permits its documented answer forms. Invalid/capped answers stay in the accuracy denominator.

This can be slow on a T4. Repeating the same command verifies complete results or generates only missing items. The old Pilot 1 baseline is preserved.

In [ ]:
from pilot_eval.sft_evaluation import evaluation_config, evaluation_directory
SFT_CONFIG, CONFIG, _, _ = restore_state()
_, baseline_config = evaluation_config(CONFIG, ARTIFACT_ROOT, 'baseline')
BASELINE_DIR = evaluation_directory(ARTIFACT_ROOT, baseline_config)
run_command(['sft-evaluate', '--config', SFT_CONFIG, '--checkpoint', 'baseline'], 'baseline', BASELINE_DIR / 'meta/status.json')
print(json.dumps(json.loads((BASELINE_DIR / 'results/results.json').read_text()), indent=2))

## 8. Train or resume the adapter

The frozen base model is unchanged. Only rank-1 LoRA parameters are trained: learning rate 1e-4, constant schedule, no warmup/dropout, effective batch 8, and 64 successful optimizer updates. With the recorded one-pass order, that is 512 example exposures.

Expected: loss/LR/gradient-norm logs, base-weight hashes and complete resumable checkpoints at 0/8/16/32/64. Each checkpoint contains optimizer, scheduler, RNG and trainer/data-position information, not just adapter weights.

After interruption, rerun this cell with the same checkout/environment/plan. It resumes from the latest **complete verified** checkpoint. Work after that checkpoint may replay (up to seven updates between checkpoints). Completed runs verify and return without retraining. Read failure logs before retrying; do not change settings in place.

In [ ]:
SFT_CONFIG, CONFIG, _, _ = restore_state()
TRAIN_DIR = training_directory(ARTIFACT_ROOT, CONFIG)
run_command(['sft-train', '--config', SFT_CONFIG], 'train', TRAIN_DIR / 'meta/status.json')
print(json.dumps(json.loads((TRAIN_DIR / 'results/results.json').read_text()), indent=2))

## 9. Evaluate each saved checkpoint

For each checkpoint, load the same untuned weights plus that checkpoint's adapter, then run the same 150 questions and decoding/scoring rules. These are separate generations because adapter weights differ.

Checkpoint zero must reproduce the matched untuned outputs. If it differs, stop and inspect the loading/evaluation conditions before interpreting later changes. Strict and flexible-v2 do **not** require two generation passes: both score each saved response.

The checkpoint runs are sequential on the single GPU. To split across sessions, choose a subset of `CHECKPOINTS_TO_RUN`; keep zero first on the initial run. Existing completed runs are checked and reused.

In [ ]:
SFT_CONFIG, CONFIG, _, _ = restore_state()
CHECKPOINTS_TO_RUN = ['0', '8', '16', '32', '64']
for checkpoint in CHECKPOINTS_TO_RUN:
    _, cell_config = evaluation_config(CONFIG, ARTIFACT_ROOT, checkpoint)
    cell_dir = evaluation_directory(ARTIFACT_ROOT, cell_config)
    run_command(['sft-evaluate', '--config', SFT_CONFIG, '--checkpoint', checkpoint],
                'evaluate-' + checkpoint, cell_dir / 'meta/status.json')

## 10. Read the paired report and trajectory

This cell is CPU-only; it performs no model inference. It verifies all six completed source cells, joins matching item IDs and reports strict/flexible-v2 accuracy, response lengths, invalid/cap rates and paired changes against the fresh baseline.

The 95% paired intervals resample the same item IDs for both models. They describe uncertainty for this run's sampled cohort, not variation across training seeds. Look at flexible accuracy and length together; shorter responses alone do not establish damage. A strict-only change can reflect output formatting.

Historical 0.640 → 0.420 and 288 → 77 values remain descriptive. The run is exploratory and has no binary collapse threshold. The report saves every paired response for inspection.

In [ ]:
SFT_CONFIG, CONFIG, _, _ = restore_state()
run_command(['sft-compare', '--config', SFT_CONFIG], 'compare')
REPORT_DIR = ARTIFACT_ROOT / 'reports' / (PLAN_NAME + '-trajectory')
from IPython.display import Markdown, SVG, display
display(Markdown((REPORT_DIR / 'results/report.md').read_text()))
display(SVG(filename=str(REPORT_DIR / 'results/trajectory.svg')))
print('Structured results:', REPORT_DIR / 'results/results.json')
print('All paired responses:', REPORT_DIR / 'results/paired-items.jsonl')

## 11. Inspect evidence and disconnect

Review a few paired responses, then use **Runtime → Disconnect and delete runtime** once GPU work is complete. Artifacts remain in Drive; CPU report generation can run later from the repo without Qwen.

Keep the original Pilot 1 runs, frozen Pilot 2 plan, training state, evaluations and report. No uploads or external Research OS updates happen automatically. Record observations separately from the frozen exploratory plan.

Future modules can read these adapters, but this pilot alone cannot identify where an adapter writes or establish a causal mechanism for any change.

In [ ]:
paired_path = REPORT_DIR / 'results/paired-items.jsonl'
shown = 0
with paired_path.open() as stream:
    for line in stream:
        pair = json.loads(line)
        if pair['step'] != 64:
            continue
        print('\nITEM:', pair['id'])
        print('BASELINE:', pair['baseline']['generated_text'])
        print('STEP 64:', pair['adapted']['generated_text'])
        print('SCORES:', pair['baseline']['score'], '→', pair['adapted']['score'])
        shown += 1
        if shown == 3:
            break
print('Training run:', TRAIN_DIR)
print('Report:', REPORT_DIR)